In [27]:
import pandas as pd
from datetime import datetime
from typing import Dict, Any, Optional

def aggregate_to_financial_analysis(
    df_income: pd.DataFrame, 
    df_expenses: pd.DataFrame, 
    account_id: str,
    reference_date: str = "2026-04-30"  # Dùng để tính active_days trong tháng
) -> Dict[str, Any]:
    """
    Aggregate dữ liệu từ Kaggle dataset cho một account (user) cụ thể
    Trả về định dạng đúng để đưa vào FinancialIntentLayer
    """
    
    # Lọc theo account
    income = df_income[df_income['account'] == account_id].copy()
    expenses = df_expenses[df_expenses['account'] == account_id].copy()
    
    if expenses.empty and income.empty:
        return {"summary": {}, "categories": {}}
    
    # Chuyển date_time sang datetime
    for df in [income, expenses]:
        if not df.empty:
            df['date'] = pd.to_datetime(df['date_time'])
    
    # ====================== SUMMARY ======================
    total_income = income['amount'].sum() if not income.empty else 0.0
    total_expense = expenses['amount'].sum() if not expenses.empty else 0.0
    balance = total_income - total_expense
    
    saving_rate = round((balance / total_income * 100), 1) if total_income > 0 else 0.0
    
    # Tính daily average (giả sử 30 ngày)
    daily_avg_expense = round(total_expense / 30, 0) if total_expense > 0 else 0
    
    # Tính std_monthly_expense (nếu có nhiều tháng)
    if not expenses.empty:
        expenses['month'] = expenses['date'].dt.to_period('M')
        monthly_exp = expenses.groupby('month')['amount'].sum()
        std_monthly = monthly_exp.std() if len(monthly_exp) > 1 else 0
    else:
        std_monthly = 0
    
    summary = {
        "total_income": round(total_income, 0),
        "total_expense": round(total_expense, 0),
        "balance": round(balance, 0),
        "saving_rate": saving_rate,
        "avg_monthly_expense": round(total_expense, 0),
        "std_monthly_expense": round(std_monthly, 0),
        "daily_avg_expense": daily_avg_expense
    }
    
    # ====================== CATEGORIES (chỉ tính chi tiêu) ======================
    categories = {}
    
    if not expenses.empty:
        cat_group = expenses.groupby('category').agg(
            amount=('amount', 'sum'),
            active_days=('date', 'nunique')
        ).reset_index()
        
        for _, row in cat_group.iterrows():
            cat_name = row['category']
            amount = row['amount']
            active_days = int(row['active_days'])
            
            pct_of_total = round((amount / total_expense * 100), 1) if total_expense > 0 else 0
            
            categories[cat_name] = {
                "pct_of_total": pct_of_total,
                "amount": round(amount, 0),
                "active_days": active_days
            }
    
    return {
        "summary": summary,
        "categories": categories
    }


# ====================== CÁCH SỬ DỤNG ======================

def get_all_users_analysis(df_income: pd.DataFrame, df_expenses: pd.DataFrame):
    """Tạo analysis cho tất cả accounts"""
    all_accounts = pd.concat([df_income['account'], df_expenses['account']]).unique()
    results = {}
    
    for acc in all_accounts:
        results[acc] = aggregate_to_financial_analysis(df_income, df_expenses, acc)
    
    return results

import numpy as np
from pprint import pformat

def clean_value(v):
    """Chuyển numpy types thành Python native và format số đẹp"""
    if isinstance(v, (np.integer, np.floating)):
        v = v.item()  # convert np.float64 / int64 thành Python float/int
    
    if isinstance(v, float):
        if v.is_integer():
            return int(v)
        else:
            return round(v, 1) if abs(v) < 1000 else round(v, 0)
    return v


def print_financial_analysis(data: dict):
    """In đẹp toàn bộ dữ liệu analysis"""
    for account, info in data.items():
        print("=" * 80)
        print(f"👤 ACCOUNT: {account}")
        print("=" * 80)
        
        summary = info.get('summary', {})
        categories = info.get('categories', {})
        
        print("📊 SUMMARY:")
        print("-" * 40)
        for key, value in summary.items():
            val = clean_value(value)
            if key == 'saving_rate':
                print(f"   {key:25}: {val:>8}%")
            elif 'amount' in key or 'expense' in key or 'balance' in key or 'income' in key:
                print(f"   {key:25}: {val:>12,} đ")
            else:
                print(f"   {key:25}: {val}")
        
        if categories:
            print("\n📂 CATEGORIES:")
            print("-" * 40)
            # Sắp xếp theo % giảm dần
            sorted_cats = sorted(
                categories.items(), 
                key=lambda x: clean_value(x[1]['pct_of_total']), 
                reverse=True
            )
            
            for cat_name, cat_info in sorted_cats:
                pct = clean_value(cat_info['pct_of_total'])
                amount = clean_value(cat_info['amount'])
                days = cat_info['active_days']
                print(f"   • {cat_name:25} | {amount:>10,} đ | {pct:5.1f}% | {days:3} ngày")
        else:
            print("\n📂 Không có chi tiêu nào.")
        
        print("\n")


In [ ]:
"""
financial_pipeline.py
─────────────────────
Pipeline hoàn chỉnh:
  1. Đọc dữ liệu thu / chi (DataFrame)
  2. Aggregate → financial_analysis dict
  3. Chạy FinancialIntentLayer → insights
  4. Lưu kết quả ra .json

Cách dùng nhanh:
    python financial_pipeline.py
  Hoặc import và gọi run_pipeline() từ code của bạn.
"""

from __future__ import annotations

import json
import os
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd


# ─────────────────────────────────────────────
# 0.  Utility helpers
# ─────────────────────────────────────────────

def _clean_value(v: Any) -> Any:
    """Chuyển numpy types → Python native, làm tròn số hợp lý."""
    if isinstance(v, (np.integer, np.floating)):
        v = v.item()
    if isinstance(v, float):
        if v.is_integer():
            return int(v)
        return round(v, 1) if abs(v) < 1_000 else round(v, 0)
    return v


def _clean_dict(d: Dict) -> Dict:
    """Đệ quy làm sạch mọi value trong dict."""
    return {k: (_clean_dict(v) if isinstance(v, dict) else _clean_value(v))
            for k, v in d.items()}


# ─────────────────────────────────────────────
# 1.  FinancialIntentLayer  (từ file của bạn)
# ─────────────────────────────────────────────

class FinancialIntentLayer:
    """
    Lớp diễn giải thông minh.
    Nhận dict số từ aggregate_to_financial_analysis() và sinh ra insights.
    """

    def __init__(self, analysis_data: Dict[str, Any]):
        self.data = analysis_data

    def detect_insights(self) -> List[str]:
        """Tự động gọi tất cả _rule_* và thu thập insights."""
        insights: List[str] = []
        for attr in sorted(dir(self)):          # sorted → thứ tự xác định
            if not attr.startswith("_rule_"):
                continue
            rule_fn = getattr(self, attr)
            if not callable(rule_fn):
                continue
            try:
                result = rule_fn()
                if result:
                    insights.append(result)
            except Exception:
                pass
        return insights

    # ── Spending patterns ─────────────────────

    def _rule_stable_food_spending(self) -> Optional[str]:
        cats = self.data.get("categories", {})
        food_keywords = ["food", "thực phẩm", "ăn", "restaurant",
                         "nhà hàng", "quán ăn", "café"]
        food_cat = next(
            (cats[n] for n in cats
             if any(kw in n.lower() for kw in food_keywords)), None
        )
        if not food_cat:
            return None
        if food_cat["pct_of_total"] > 20 and food_cat["active_days"] > 20:
            return (
                f"✓ Chi tiêu thức ăn: {food_cat['pct_of_total']}% tổng chi, "
                f"phân bổ đều {food_cat['active_days']}/30 ngày → thói quen ổn định, "
                f"nhu cầu cơ bản được đáp ứng."
            )
        return None

    def _rule_impulsive_shopping(self) -> Optional[str]:
        cats = self.data.get("categories", {})
        shopping_keywords = ["shopping", "mua sắm", "fashion", "clothes",
                             "thời trang", "quần áo", "shoes", "giày"]
        shopping_cat = next(
            (cats[n] for n in cats
             if any(kw in n.lower() for kw in shopping_keywords)), None
        )
        if not shopping_cat:
            return None
        if shopping_cat["pct_of_total"] > 20 and shopping_cat["active_days"] < 8:
            return (
                f"⚠️ Mua sắm: {shopping_cat['pct_of_total']}% tổng chi nhưng chỉ "
                f"{shopping_cat['active_days']} ngày → chi tiêu tập trung không kiểm soát. "
                f"Gợi ý: lập kế hoạch chi tiêu trước."
            )
        return None

    def _rule_entertainment_frequency(self) -> Optional[str]:
        cats = self.data.get("categories", {})
        ent_keywords = ["entertainment", "giải trí", "movie", "phim",
                        "game", "trò chơi", "concert", "bar", "pub"]
        ent_cat = next(
            (cats[n] for n in cats
             if any(kw in n.lower() for kw in ent_keywords)), None
        )
        if not ent_cat:
            return None
        if ent_cat["pct_of_total"] > 10 and ent_cat["active_days"] > 15:
            return (
                f"ℹ️ Giải trí: {ent_cat['pct_of_total']}% tổng chi, "
                f"{ent_cat['active_days']} ngày → bạn có thói quen thường xuyên. "
                f"Đây có thể là dấu hiệu stress."
            )
        return None

    def _rule_transport_efficiency(self) -> Optional[str]:
        cats = self.data.get("categories", {})
        transport_keywords = ["transport", "vận chuyển", "taxi", "grab",
                              "bus", "xăng", "petrol", "gas", "xe"]
        transport_cat = next(
            (cats[n] for n in cats
             if any(kw in n.lower() for kw in transport_keywords)), None
        )
        if not transport_cat:
            return None
        if transport_cat["pct_of_total"] > 15 and transport_cat["active_days"] > 20:
            return (
                f"💡 Vận chuyển: {transport_cat['pct_of_total']}% tổng chi → xem xét "
                f"công ty cấp xe, đi bộ, hay dùng phương tiện công cộng."
            )
        return None

    # ── Saving rate ───────────────────────────

    def _rule_critical_saving_rate(self) -> Optional[str]:
        saving = self.data["summary"]["saving_rate"]
        if saving < 5:
            return (
                f"🚨 CẢNH BÁO: Tỷ lệ tiết kiệm chỉ {saving}% (gần như không tiết kiệm). "
                f"Bạn cần giảm chi tiêu ngay để tránh nợ nần."
            )
        return None

    def _rule_low_saving_rate(self) -> Optional[str]:
        saving = self.data["summary"]["saving_rate"]
        if 5 <= saving <= 15:
            return (
                f"🟡 Tỷ lệ tiết kiệm {saving}% là thấp. "
                f"Mục tiêu nên là 20-30%. Hãy xem lại các khoản chi không cần thiết."
            )
        return None

    def _rule_good_saving_rate(self) -> Optional[str]:
        saving = self.data["summary"]["saving_rate"]
        if 15 < saving <= 30:
            return f"✅ Tỷ lệ tiết kiệm {saving}% là tốt. Tiếp tục duy trì!"
        return None

    def _rule_excellent_saving_rate(self) -> Optional[str]:
        saving = self.data["summary"]["saving_rate"]
        if saving > 30:
            return (
                f"🌟 Tỷ lệ tiết kiệm {saving}% là xuất sắc! "
                f"Bạn kiểm soát chi tiêu rất tốt."
            )
        return None

    # ── Spending volatility ───────────────────

    def _rule_high_spending_volatility(self) -> Optional[str]:
        summary = self.data["summary"]
        avg, std = summary["avg_monthly_expense"], summary["std_monthly_expense"]
        if avg > 0 and (std / avg) > 0.3:
            return (
                f"📊 Chi tiêu không ổn định: độ lệch {std:,.0f}đ "
                f"(tương đối {std / avg * 100:.0f}%). "
                f"Gợi ý: lập ngân sách cụ thể cho từng tháng."
            )
        return None

    def _rule_stable_spending_pattern(self) -> Optional[str]:
        summary = self.data["summary"]
        avg, std = summary["avg_monthly_expense"], summary["std_monthly_expense"]
        if avg > 0 and (std / avg) < 0.15:
            return (
                f"✓ Chi tiêu ổn định: chênh lệch giữa các tháng chỉ {std:,.0f}đ → "
                f"bạn có thói quen chi tiêu dự đoán được."
            )
        return None

    # ── Income vs expense ─────────────────────

    def _rule_deficit_spending(self) -> Optional[str]:
        balance = self.data["summary"]["balance"]
        income  = self.data["summary"]["total_income"]
        if balance < 0:
            deficit = abs(balance)
            return (
                f"🔴 CHỈ SỐ NGUY: Chi tiêu vượt thu nhập {deficit:,.0f}đ "
                f"({-balance / income * 100:.1f}%). Bạn đang sống vượt khả năng!"
            )
        return None

    def _rule_healthy_balance(self) -> Optional[str]:
        balance = self.data["summary"]["balance"]
        income  = self.data["summary"]["total_income"]
        if 0 < balance <= income * 0.3:
            return (
                f"✓ Tình hình tài chính khỏe mạnh: dư {balance:,.0f}đ. "
                f"Tiếp tục duy trì!"
            )
        return None

    # ── Category balance ──────────────────────

    def _rule_top_spending_category(self) -> Optional[str]:
        cats = self.data.get("categories", {})
        if not cats:
            return None
        cat_name, cat_info = max(cats.items(), key=lambda x: x[1]["pct_of_total"])
        if cat_info["pct_of_total"] > 25:
            return (
                f"📌 {cat_name} là khoản chi cao nhất: {cat_info['pct_of_total']}% tổng chi. "
                f"Cân nhắc xem có thể tiết kiệm ở mục này không."
            )
        return None

    def _rule_diversified_spending(self) -> Optional[str]:
        cats = self.data.get("categories", {})
        if len(cats) < 3:
            return None
        max_pct = max(c["pct_of_total"] for c in cats.values())
        if max_pct < 30:
            return (
                f"✓ Chi tiêu của bạn đa dạng trên {len(cats)} danh mục. "
                f"Điều này giúp giảm rủi ro."
            )
        return None

    # ── Daily average ─────────────────────────

    def _rule_daily_spending_average(self) -> Optional[str]:
        daily_avg = self.data["summary"]["daily_avg_expense"]
        if daily_avg > 500_000:
            return (
                f"📈 Mức chi tiêu trung bình {daily_avg:,.0f}đ/ngày. "
                f"Hãy xem có khoản chi không cần thiết không."
            )
        if daily_avg < 100_000:
            return (
                f"✓ Mức chi tiêu trung bình chỉ {daily_avg:,.0f}đ/ngày → rất tiết kiệm."
            )
        return None


# ─────────────────────────────────────────────
# 2.  Aggregation  (từ code của bạn, đã refactor)
# ─────────────────────────────────────────────

def aggregate_to_financial_analysis(
    df_income: pd.DataFrame,
    df_expenses: pd.DataFrame,
    account_id: str,
    reference_date: str = "2026-04-30",
) -> Dict[str, Any]:
    """
    Aggregate dữ liệu thu/chi của một account thành dict chuẩn
    để đưa vào FinancialIntentLayer.

    Parameters
    ----------
    df_income    : DataFrame có cột [account, date_time, amount]
    df_expenses  : DataFrame có cột [account, date_time, amount, category]
    account_id   : ID của tài khoản cần phân tích
    reference_date : mốc thời gian tham chiếu (chưa dùng nhưng giữ để mở rộng)
    """
    income   = df_income[df_income["account"] == account_id].copy()
    expenses = df_expenses[df_expenses["account"] == account_id].copy()

    if income.empty and expenses.empty:
        return {"summary": {}, "categories": {}}

    # Parse date
    for df in [income, expenses]:
        if not df.empty:
            df["date"] = pd.to_datetime(df["date_time"])

    # ── Summary ──────────────────────────────
    total_income  = float(income["amount"].sum())  if not income.empty  else 0.0
    total_expense = float(expenses["amount"].sum()) if not expenses.empty else 0.0
    balance       = total_income - total_expense
    saving_rate   = round(balance / total_income * 100, 1) if total_income > 0 else 0.0
    daily_avg     = round(total_expense / 30, 0) if total_expense > 0 else 0.0

    if not expenses.empty:
        expenses["month"] = expenses["date"].dt.to_period("M")
        monthly_exp = expenses.groupby("month")["amount"].sum()
        avg_monthly = float(monthly_exp.mean()) if len(monthly_exp) > 0 else total_expense
        std_monthly = float(monthly_exp.std())  if len(monthly_exp) > 1 else 0.0
    else:
        avg_monthly = total_expense
        std_monthly = 0.0

    summary = {
        "total_income":        round(total_income,  0),
        "total_expense":       round(total_expense, 0),
        "balance":             round(balance,        0),
        "saving_rate":         saving_rate,
        "avg_monthly_expense": round(avg_monthly,   0),
        "std_monthly_expense": round(std_monthly,   0),
        "daily_avg_expense":   round(daily_avg,     0),
    }

    # ── Categories (chỉ chi tiêu) ────────────
    categories: Dict[str, Any] = {}
    if not expenses.empty:
        cat_group = (
            expenses.groupby("category")
            .agg(amount=("amount", "sum"), active_days=("date", "nunique"))
            .reset_index()
        )
        for _, row in cat_group.iterrows():
            pct = round(row["amount"] / total_expense * 100, 1) if total_expense > 0 else 0.0
            categories[row["category"]] = {
                "amount":      round(float(row["amount"]), 0),
                "pct_of_total": pct,
                "active_days": int(row["active_days"]),
            }

    return {"summary": summary, "categories": categories}


def get_all_users_analysis(
    df_income: pd.DataFrame,
    df_expenses: pd.DataFrame,
) -> Dict[str, Any]:
    """Tạo analysis dict cho tất cả accounts."""
    all_accounts = pd.concat(
        [df_income["account"], df_expenses["account"]]
    ).unique()
    return {
        acc: aggregate_to_financial_analysis(df_income, df_expenses, acc)
        for acc in all_accounts
    }


# ─────────────────────────────────────────────
# 3.  Pipeline chính
# ─────────────────────────────────────────────

def run_pipeline(
    df_income: pd.DataFrame,
    df_expenses: pd.DataFrame,
    output_dir: str = ".",
    single_account: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Chạy toàn bộ pipeline cho một hoặc tất cả accounts.

    Parameters
    ----------
    df_income       : DataFrame thu nhập
    df_expenses     : DataFrame chi tiêu
    output_dir      : thư mục lưu file JSON
    single_account  : nếu chỉ muốn xử lý 1 account; None = xử lý tất cả

    Returns
    -------
    results : dict  {account_id: {financial_summary, insights}}
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Lấy danh sách account cần xử lý
    if single_account:
        accounts = [single_account]
    else:
        accounts = pd.concat(
            [df_income["account"], df_expenses["account"]]
        ).unique().tolist()

    results: Dict[str, Any] = {}

    for acc in accounts:
        # 1) Aggregate
        analysis = aggregate_to_financial_analysis(df_income, df_expenses, acc)
        if not analysis.get("summary"):
            print(f"[SKIP] {acc}: không có dữ liệu.")
            continue

        # 2) Detect insights
        intent_layer = FinancialIntentLayer(analysis)
        insights = intent_layer.detect_insights()

        # 3) Build output record
        record = {
            "account_id":        acc,
            "generated_at":      datetime.now().isoformat(timespec="seconds"),
            "financial_summary": _clean_dict(analysis["summary"]),
            "categories":        _clean_dict(analysis["categories"]),
            "insights":          insights,
        }

        # 4) Lưu file JSON riêng cho từng account
        json_path = Path(output_dir) / f"{acc}_financial_insights.json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)

        print(f"[OK] {acc} → {json_path}  ({len(insights)} insights)")
        results[acc] = record

    # Lưu thêm file tổng hợp tất cả accounts
    all_path = Path(output_dir) / "all_accounts_insights.json"
    with open(all_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\n✅ Tổng hợp tất cả → {all_path}")

    return results


if __name__ == "__main__":
    df_income = pd.read_csv('./data/Income_clean.csv')
    df_expenses = pd.read_csv('./data/Expenses_clean.csv')

    results = run_pipeline(
        df_income=df_income,
        df_expenses=df_expenses,
        output_dir="financial_outputs",   # ← thư mục xuất file
    )

    # In thử kết quả của account đầu tiên
    first_acc = next(iter(results))
    print("\n─── Preview: " + first_acc + " ───")
    print(json.dumps(results[first_acc], ensure_ascii=False, indent=2))

[OK] acct_1 → financial_outputs/acct_1_financial_insights.json  (4 insights)
[OK] acct_2 → financial_outputs/acct_2_financial_insights.json  (6 insights)
[OK] acct_3 → financial_outputs/acct_3_financial_insights.json  (5 insights)
[OK] acct_4 → financial_outputs/acct_4_financial_insights.json  (2 insights)
[OK] acct_5 → financial_outputs/acct_5_financial_insights.json  (2 insights)

✅ Tổng hợp tất cả → financial_outputs/all_accounts_insights.json

─── Preview: acct_1 ───
{
  "account_id": "acct_1",
  "generated_at": "2026-05-16T00:16:04",
  "financial_summary": {
    "total_income": 24936,
    "total_expense": 13064,
    "balance": 11872,
    "saving_rate": 47.6,
    "avg_monthly_expense": 1188,
    "std_monthly_expense": 902,
    "daily_avg_expense": 435
  },
  "categories": {
    "Bought for myself": {
      "amount": 1273,
      "pct_of_total": 9.7,
      "active_days": 4
    },
    "Cafe": {
      "amount": 1109,
      "pct_of_total": 8.5,
      "active_days": 85
    },
    "Clothe

In [ ]:
income_df = pd.read_csv('./data/Income_clean.csv')
expenses_df = pd.read_csv('./data/Expenses_clean.csv')

print("="*20 + " INCOME " + "="*20)
income_df.info()
print("\n\n" + "="*20 + " EXPENSES " + "="*20)
expenses_df.info()

In [28]:
analysis_data = get_all_users_analysis(income_df, expenses_df)
print_financial_analysis(analysis_data)

👤 ACCOUNT: acct_1
📊 SUMMARY:
----------------------------------------
   total_income             :       24,936 đ
   total_expense            :       13,064 đ
   balance                  :       11,872 đ
   saving_rate              :     47.6%
   avg_monthly_expense      :       13,064 đ
   std_monthly_expense      :          902 đ
   daily_avg_expense        :          435 đ

📂 CATEGORIES:
----------------------------------------
   • Loan given                |      2,809 đ |  21.5% |  10 ngày
   • Other                     |      2,465 đ |  18.9% |  11 ngày
   • Food                      |      1,322 đ |  10.1% |  72 ngày
   • Bought for myself         |      1,273 đ |   9.7% |   4 ngày
   • Gifts                     |      1,177 đ |   9.0% |  29 ngày
   • Cafe                      |      1,109 đ |   8.5% |  85 ngày
   • Health                    |        772 đ |   5.9% |  18 ngày
   • Leisure                   |        738 đ |   5.6% |  22 ngày
   • Taxi                      |    